# 第7章 評価と運用 実践

ローカル LLM は、プロンプト、RAG、LoRA、追加学習のどれを変えても、同じ評価ケースで見直せるようにしておくことが大切です。この章では、短い評価ケースを実行し、最低限の回帰確認をします。


In [ ]:
from pathlib import Path
import os
import json
import sys

# Notebook をどこから開いても helper を import できるようにします。
search_roots = [Path.cwd()]
env_root = os.environ.get("LOCAL_LLM_REPO_ROOT")
if env_root:
    search_roots.append(Path(env_root))
search_roots.append(Path("C:/LLM"))

seen = set()
for root in search_roots:
    current = root.resolve()
    for candidate in [current, *current.parents]:
        if candidate in seen:
            continue
        seen.add(candidate)
        helper_dir = candidate / "notebooks"
        if (helper_dir / "local_llm_practice.py").exists():
            sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise RuntimeError(
        "notebooks/local_llm_practice.py が見つかりません。"
        "C:/LLM か notebooks/ 配下で開くか、LOCAL_LLM_REPO_ROOT を設定してください。"
    )

from local_llm_practice import (
    DATA_DIR,
    DOCS_DIR,
    REPO_ROOT,
    WORK_DIR,
    TrainingConfig,
    ask_about_chapter,
    attach_lora,
    configure_local_caches,
    count_trainable_parameters,
    generate_text,
    gpu_summary,
    load_base_model,
    load_chapter,
    load_jsonl,
    make_cpt_features,
    make_sft_features,
    nvidia_smi_summary,
    ollama_generate,
    print_headings,
    read_text,
    retrieve_chunks,
    split_markdown,
    train_lora_adapter,
)

configure_local_caches()
print("REPO_ROOT:", REPO_ROOT)
print("DOCS_DIR :", DOCS_DIR)
print("WORK_DIR :", WORK_DIR)

chapter_path, chapter_text = load_chapter("07-evaluation-operation.md")
print(chapter_path)
print_headings(chapter_text)


## 1. 評価ケースを作る

`must_include` は機械的な最低条件です。これだけで品質を保証するものではありませんが、回帰確認の入口になります。


In [ ]:
evaluation_cases = [
    {"id": "setup-001", "question": "Ollama が応答しない時、最初に確認することは？", "must_include": ["起動", "モデル", "接続"]},
    {"id": "rag-001", "question": "RAG の回答で出典を確認する理由は？", "must_include": ["根拠", "確認"]},
    {"id": "lora-001", "question": "LoRA を知識の丸暗記に使いすぎない方がよい理由は？", "must_include": ["形式", "RAG"]},
]
print(json.dumps(evaluation_cases, ensure_ascii=False, indent=2))


## 2. ローカル LLM に回答させる


In [ ]:
results = []
for case in evaluation_cases:
    prompt = f"""
あなたはローカル LLM 入門教材のチューターです。
質問に5文以内で答えてください。

質問: {case['question']}
""".strip()
    response = ollama_generate(prompt, temperature=0.2)
    results.append({**case, "response": response})
    print(f"\n=== {case['id']} ===")
    print(response)


## 3. 最低限の回帰確認をする


In [ ]:
scores = []
for result in results:
    checks = {keyword: (keyword in result["response"]) for keyword in result["must_include"]}
    scores.append({"id": result["id"], "passed": all(checks.values()), "checks": checks})
print(json.dumps(scores, ensure_ascii=False, indent=2))


## 4. 人間が見る観点を追加する

自動チェックは、根拠の自然さ、危ない断定、説明の読みやすさを十分に評価できません。最後は人間が見る観点を明文化します。


In [ ]:
print(ask_about_chapter(chapter_text, "自動評価だけでは見落としやすい観点を、運用チェックリストとしてまとめてください。"))
